In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pathlib import Path
import os
import pyspark
from dotenv import load_dotenv


In [2]:
spark = SparkSession.builder.appName('Viagem').getOrCreate()

In [3]:
caminho_env = Path.cwd() / '.env'
load_dotenv(caminho_env)

True

In [4]:
df_viagem = (
    spark.read
    .format('json')
    .option('multiline',True)
    .option('timestampFormat',"yyyy-MM-dd'T'HH:mm:ss")
    .option("columnNameOfCorruptRecord","_dados_corrompidos")
    .load(os.getenv('VIAGEM'))
)

In [5]:
df_viagem = df_viagem.withColumns({
    'entregas': explode(col('entregas')),
    'data_inicio' : date_format(to_date(col('data_inicio'),"yyyy-MM-dd'T'HH:mm:ss"),"yyyy-MM-dd")
}).select('data_inicio','entregas.*','id_viagem','motorista.*','veiculo.*')

In [6]:
df_viagem = df_viagem.withColumn(
    'data_inicio',to_date(col('data_inicio'))
)

In [7]:
df_viagem = df_viagem.withColumns({
    'capacidade_toneladas':col('capacidade_toneladas').cast('int'),
    'valor_frete': col('valor_frete').cast('float')
})

In [8]:
df_viagem.show(1)

+-----------+--------------+--------+--------------+----------+-----------+---------+--------------+----------+--------------------+--------+---------------+
|data_inicio|cidade_destino| cliente|id_nota_fiscal|status_sla|valor_frete|id_viagem|           cpf|      nome|capacidade_toneladas|   placa|           tipo|
+-----------+--------------+--------+--------------+----------+-----------+---------+--------------+----------+--------------------+--------+---------------+
| 2026-11-14|      Curitiba|Quimisul|       NF-9001|  No Prazo|     1500.5|   V-1001|111.222.333-44|João Silva|                  25|ABC-1234|Carreta 3 Eixos|
+-----------+--------------+--------+--------------+----------+-----------+---------+--------------+----------+--------------------+--------+---------------+
only showing top 1 row


In [9]:
df_viagem.printSchema()

root
 |-- data_inicio: date (nullable = true)
 |-- cidade_destino: string (nullable = true)
 |-- cliente: string (nullable = true)
 |-- id_nota_fiscal: string (nullable = true)
 |-- status_sla: string (nullable = true)
 |-- valor_frete: float (nullable = true)
 |-- id_viagem: string (nullable = true)
 |-- cpf: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- capacidade_toneladas: integer (nullable = true)
 |-- placa: string (nullable = true)
 |-- tipo: string (nullable = true)

